
# 12 — Data Imputation and Encoding

**Scope:** The missing and incorrect values are handled, categorical features are encoded.


Since we decided to predict missing and incorrect values, we have to split the training data into training and validation sets before applying imputation and encoding.

# Table of Contents

Take this as an example for a Table of Contents for your notebook.
We have to fix all the names and sections according to what we actually do in the notebook.

<a class="anchor" id="top"></a>

** **

1. [Importing Libraries & Data](#1.-Importing-Libraries-&-Data) <br><br>
    
2. [Exploratory Data Analysis](#2.-Exploratory-Data-Analysis)
    
   2.1 [Incoherencies](#2.1-Incoherencies) <br>
   
   &emsp; 2.1.1 [Address Identified Incoherencies](#2.1.1-Address-Identified-Incoherencies) <br><br>
    
3. [Data Cleaning & Preprocessing](#3.-Data-Cleaning-&-Preprocessing)

   3.1 [Duplicates](#3.1-Duplicates) <br>
    
   3.2 [Feature Engineering](#3.2-Feature-Engineering) <br>
   
   &emsp; 3.2.1 [Data Type Conversions](#3.2.1-Data-Type-Conversions) <br>
   
   &emsp; 3.2.2 [Encoding](#3.2.2-Encoding) <br>
   
   &emsp; 3.2.3 [Other Transformations](#3.2.3-Other-Transformations) <br>
    
   &emsp; 3.2.4 [Unique Feature-Pair Analysis](#3.2.4-Unique-Feature-Pair-Analysis) <br> 

   3.3 [Train-Test Split](#3.3-Train-Test-Split) <br>
   
   3.4 [Missing Values](#3.4-Missing-Values) <br>
    
   3.5 [Outliers](#3.5-Outliers) <br>

   3.6 [Visualisations](#3.6-Visualisations) <br><br>
   

In [1]:
import os, re, math, warnings
from pathlib import Path
from datetime import datetime
import json
import pandas as pd
import numpy as np
import re
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import RobustScaler

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)
pd.set_option("mode.copy_on_write", True)
warnings.filterwarnings("ignore")

RANDOM_STATE = 42  # for reproducibility of any sampling


In [2]:
# Load the data paths
data_dir = "../data/"

# Load the raw data into a pandas dataframe
df = pd.read_csv(os.path.join(data_dir, "processed_data/11_processed_train_data.csv"))
X_test = pd.read_csv(os.path.join(data_dir, "processed_data/11_processed_test_data.csv"))

# put carID as Index
df.set_index("carID", inplace=True)
X_test.set_index("carID", inplace=True)

print("Loaded shape:", df.shape)
display(df.head(3))

print("Loaded test shape:", X_test.shape)
display(X_test.head(3))


Loaded shape: (74467, 13)


,Brand,model,year,price,transmission,mileage,fuelType,tax,mpg,engineSize,paintQuality%,previousOwners,hasDamage
carID,,,,,,,,,,,,,
69512,Volkswagen,Golf,2016.0,22290.0,Semi-Auto,28421.0,Petrol,NaN,11.417268,2.0,63.0,4.0,0.0
53000,Toyota,Yaris,2019.0,13790.0,Manual,4589.0,Petrol,145.0,47.900000,1.5,50.0,1.0,0.0
6366,Audi,Q2,2019.0,24990.0,Semi-Auto,3624.0,Petrol,145.0,40.900000,1.5,56.0,4.0,0.0


Loaded test shape: (32567, 12)


,Brand,model,year,transmission,mileage,fuelType,tax,mpg,engineSize,paintQuality%,previousOwners,hasDamage
carID,,,,,,,,,,,,
89856,Hyundai,i30,NaN,Automatic,30700.0,Petrol,205.0,41.5,1.6,61.0,3.0,0.0
106581,Volkswagen,Tiguan,2017.0,Semi-Auto,NaN,Petrol,150.0,38.2,2.0,60.0,2.0,0.0
80886,BMW,2 Series,2016.0,Automatic,36792.0,Petrol,125.0,51.4,1.5,94.0,2.0,0.0


### Split the Training data into training and validation sets

In [3]:
# ======================================================
# Split df into Training and Validation Set (60/20/20 total)
# ======================================================

from sklearn.model_selection import train_test_split

# Separate features and target from the training data
X = df.drop(columns=["price"])
y = df["price"]

# Split df (80% of total data) into training and validation
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.25,        # 25% of 80% train = 20% of total
    random_state=42,
)

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_val shape:   {X_val.shape}")
print(f"y_val shape:   {y_val.shape}")
print(f"X_test shape:  {X_test.shape}")


X_train shape: (55850, 12)
y_train shape: (55850,)
X_val shape:   (18617, 12)
y_val shape:   (18617,)
X_test shape:  (32567, 12)


# Handling Missing Values

- Numerical: fill with the median
- Categorical: fill with the most frequent value

In [4]:
import pandas as pd

def missing_report(X: pd.DataFrame, name: str) -> pd.DataFrame:
    mv = X.isna().sum()
    mv = mv[mv > 0].sort_values(ascending=False)
    if mv.empty:
        print(f"[{name}] No missing values found. (n_rows={len(X)})")
        return pd.DataFrame(columns=["n_missing", "pct_missing"])
    pct = (mv / len(X) * 100).round(2)
    report = pd.DataFrame({"n_missing": mv, "pct_missing": pct})
    print(f"[{name}] Missing Values (n_rows={len(X)}):")
    display(report)
    return report

# Reports for the splits
mv_train = missing_report(X_train, "X_train")
mv_val   = missing_report(X_val,   "X_val")
mv_test  = missing_report(X_test,  "X_test")

# Helpful for the next step (imputation/encoding):
numeric_cols = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = X_train.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

print(f"#numeric_cols: {len(numeric_cols)} → {numeric_cols[:10]}{' ...' if len(numeric_cols) > 10 else ''}")
print(f"#categorical_cols: {len(categorical_cols)} → {categorical_cols[:10]}{' ...' if len(categorical_cols) > 10 else ''}")


[X_train] Missing Values (n_rows=55850):


,n_missing,pct_missing
mpg,6750,12.09
tax,6079,10.88
transmission,1666,2.98
engineSize,1548,2.77
previousOwners,1425,2.55
mileage,1380,2.47
paintQuality%,1378,2.47
model,1228,2.20
hasDamage,1136,2.03
fuelType,1113,1.99


[X_val] Missing Values (n_rows=18617):


,n_missing,pct_missing
mpg,2254,12.11
tax,2029,10.90
transmission,552,2.97
engineSize,507,2.72
paintQuality%,475,2.55
previousOwners,463,2.49
model,425,2.28
mileage,421,2.26
hasDamage,385,2.07
fuelType,366,1.97


[X_test] Missing Values (n_rows=32567):


,n_missing,pct_missing
mpg,3840,11.79
tax,3469,10.65
year,1007,3.09
transmission,968,2.97
previousOwners,936,2.87
engineSize,875,2.69
mileage,859,2.64
paintQuality%,793,2.43
model,734,2.25
fuelType,656,2.01


#numeric_cols: 8 → ['year', 'mileage', 'tax', 'mpg', 'engineSize', 'paintQuality%', 'previousOwners', 'hasDamage']
#categorical_cols: 4 → ['Brand', 'model', 'transmission', 'fuelType']


Why we chose random forest for imputation of 'transmission' feature? 

tbd

In [5]:
def impute_transmission(df, min_model_count=10):
    # Make a copy so the original dataframe isn't modified directly
    df = df.copy()
    # Count how many entries exist per model
    # We drop rows where with nan for this count, because we dont want data leakage
    df_nona = df.copy().dropna()
    model_counts = df_nona['model'].value_counts()

    # Keep only models with enough data points
    valid_models = model_counts[model_counts >= min_model_count].index

    # Loop through those models and fill NaN with the most common value (mode)
    for model in valid_models:
        mask = (df['model'] == model) & (df['transmission'].isna())
        mode_values = df.loc[df['model'] == model, 'transmission'].mode()
        if not mode_values.empty:
            df.loc[mask, 'transmission'] = mode_values[0]

    # Check how many missing values are still left
    remaining_nas = df['transmission'].isna().sum()
    
    if remaining_nas > 0:
        # Encode categorical variables numerically
        # RandomForest can only work with numeric data
        for col in ['Brand', 'model', 'fuelType']:
            without_na = df.dropna()
            counts = without_na[col].value_counts()
            df[col] = df[col].map(counts).astype(float)

        # Split into training (known transmission) and test (missing transmission)
        transmission_train = df[df['transmission'].notna()]
        transmission_test = df[df['transmission'].isna()]

        # Select predictor features
        features = ['Brand', 'model', 'fuelType', 'engineSize', 'year', 'mpg', 'tax']
        X_train = transmission_train[features]
        y_train = transmission_train['transmission']
        X_test = transmission_test[features]

        # Train a Random Forest classifier to predict transmission type
        model = RandomForestClassifier(
            n_estimators=200,  # number of trees
            max_depth=10,      # limit depth to avoid overfitting
            random_state=42
        )
        model.fit(X_train, y_train)

        # Predict missing transmission values
        preds = model.predict(X_test)
        df.loc[df['transmission'].isna(), 'transmission'] = preds
        return df

In [6]:
def impute_fuelType(df, min_model_count=10):
    
    # work on a copy
    df = df.copy()

    # per-model mode fill (only for models with enough rows)
    # We drop rows where with nan for this count, because we dont want data leakage
    df_nona = df.copy().dropna()
    model_counts = df_nona['model'].value_counts()
    valid_models = model_counts[model_counts >= min_model_count].index

    for m in valid_models:
        mask_missing = (df['model'] == m) & (df['fuelType'].isna())
        mode_vals = df.loc[df['model'] == m, 'fuelType'].mode()
        if not mode_vals.empty:
            df.loc[mask_missing, 'fuelType'] = mode_vals[0]

    # if still missing, train a classifier
    remaining = df['fuelType'].isna().sum()
    if remaining > 0:
        # encode categorical predictors
        predictors = ['Brand', 'model', 'transmission', 'engineSize', 'year', 'mpg']
        for col in predictors:
            if df[col].dtype == 'object':
                without_na = df.dropna()
                counts = without_na[col].value_counts()
                df[col] = df[col].map(counts).astype(float)

        # split into known vs missing target
        fuelType_train = df[df['fuelType'].notna()]
        test  = df[df['fuelType'].isna()]

        X_train = fuelType_train[predictors]
        y_train = fuelType_train['fuelType']
        X_test  = test[predictors]

        # encode target
        y_le = LabelEncoder()
        y_train_enc = y_le.fit_transform(y_train.astype(str))

        # train classifier
        clf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=77)
        clf.fit(X_train, y_train_enc)

        # predict and inverse-transform
        preds_enc = clf.predict(X_test)
        preds = y_le.inverse_transform(preds_enc)

        # fill back
        df.loc[df['fuelType'].isna(), 'fuelType'] = preds

    return df

In [7]:
def impute_numeric_rf(df, target_col, rf_features):
    """
    Fill missing numeric values in a column using a RandomForestRegressor.
    """

    # Copy the DataFrame so the original one is not changed
    df = df.copy()

    # If there are no missing values, just return the DataFrame
    if not df[target_col].isna().any():
        return df

    # Encode categorical features so the model can handle them
    for col in rf_features:
        if df[col].dtype == 'object':
            without_na = df.dropna()
            counts = without_na[col].value_counts()
            df[col] = df[col].map(counts).astype(float)
    # Split data into known and missing target values
    train = df[df[target_col].notna()]
    test = df[df[target_col].isna()]

    X_train = train[rf_features]
    y_train = train[target_col]
    X_test = test[rf_features]

    # Train the Random Forest model
    # n_estimators=300 number of trees in the forest, more trees → better accuracy, but slower
    # max_depth=12 maximum depth of each tree, limits how detailed (complex) each tree can get → prevents overfitting
    # random_state=23 seed for randomness, makes results reproducible (same every time you run it)
    model = RandomForestRegressor(n_estimators=300, max_depth=12, random_state=23)
    model.fit(X_train, y_train)

    # Predict missing values and fill them in
    preds = model.predict(X_test)
    df.loc[df[target_col].isna(), target_col] = preds

    return df

In [8]:
def impute_mileage(df):
    # Make a copy so the original DataFrame is not modified
    df = df.copy()

    # Calculate the mean mileage for each year (only using non-missing values)
    year_medians = df.groupby('year')['mileage'].median()

    # Replace missing mileage values with the corresponding year mean
    df['mileage'] = df['mileage'].fillna(df['year'].map(year_medians))
    df['mileage'] = df['mileage'].fillna(df['mileage'].median())
    # Return the DataFrame with imputed values
    return df

In [9]:
def impute_year(df, bins=30):

    # Make a copy to avoid modifying the original DataFrame
    df = df.copy()

    # Create mileage bins
    df['mileage_bin'] = pd.cut(df['mileage'], bins=bins)

    # Calculate mean year per mileage bin (only where year is known)
    bin_medians = df.groupby('mileage_bin')['year'].median().round()

    # Map each row’s bin to its mean year
    df['year'] = df['year'].fillna(df['mileage_bin'].map(bin_medians))

    df['year'] = df['year'].fillna(df['year'].median())

    # Drop the helper column
    df.drop(columns='mileage_bin', inplace=True)

    return df

In [10]:
def impute_mpg(df):
    # Make a copy so the original DataFrame is not modified
    df = df.copy()

     # Calculate median mpg by (model, fuelType)
    model_medians = df.groupby(['model', 'fuelType'])['mpg'].median()

    # Calculate fallback median mpg by (Brand, fuelType)
    brand_medians = df.groupby(['Brand', 'fuelType'])['mpg'].median()

    # Calculate fallback median mpg by fuelType
    fuelType_medians = df.groupby(['fuelType'])['mpg'].median()

    # Iterate through missing rows and fill based on available group
    for idx, row in df[df['mpg'].isna()].iterrows():
        key_model = (row['model'], row['fuelType'])
        key_brand = (row['Brand'], row['fuelType'])

        if key_model in model_medians.index:
            df.at[idx, 'mpg'] = model_medians.loc[key_model]
        elif key_brand in brand_medians.index:
            df.at[idx, 'mpg'] = brand_medians.loc[key_brand]
    df['mpg'] = df['mpg'].fillna(df['fuelType'].map(fuelType_medians))
    df['mpg'] = df['mpg'].fillna(df['mpg'].median())

    # Return the DataFrame with imputed values
    return df

In [11]:
def impute_paintQuality(df):

    df = df.copy()

    median_val = df['paintQuality%'].median()
    df['paintQuality%'] = df['paintQuality%'].fillna(median_val)    

    return df

In [12]:
def impute_previousOwners(df):

    df = df.copy()

    median_val = df['previousOwners'].median()
    df['previousOwners'] = df['previousOwners'].fillna(median_val)    

    return df

In [13]:
def impute_tax(df, bins=15):
    # Make a copy so the original DataFrame is not modified
    df = df.copy()
    df['mpg_bin'] = pd.cut(df['mpg'], bins=bins)
    
     # Calculate median tax by (model, fuelType)
    medians_one = df.groupby(['model', 'fuelType', 'mpg_bin'])['tax'].median()

    # Calculate fallback median tax by (Brand, fuelType)
    medians_two = df.groupby(['Brand', 'fuelType', 'mpg_bin'])['tax'].median()

    # Calculate fallback median tax by fuelType
    medians_three = df.groupby(['fuelType', 'mpg_bin'])['tax'].median()

    # Iterate through missing rows and fill based on available group
    for idx, row in df[df['tax'].isna()].iterrows():
        key_model = (row['model'], row['fuelType'], row['mpg_bin'])
        key_brand = (row['Brand'], row['fuelType'], row['mpg_bin'])
        key_else =  (row['fuelType'], row['mpg_bin'])

        if key_model in medians_one.index:
            df.at[idx, 'tax'] = medians_one.loc[key_model]
        elif key_brand in medians_two.index:
            df.at[idx, 'tax'] = medians_two.loc[key_brand]
        elif key_else in medians_three.index:
            df.at[idx, 'tax'] = medians_three.loc[key_else]
    fuelType_medians = df.groupby('fuelType')['tax'].median()

    # Replace missing mileage values with the corresponding year mean
    df['tax'] = df['tax'].fillna(df['fuelType'].map(fuelType_medians))
    df['tax'] = df['tax'].fillna(df['tax'].median())

    # Return the DataFrame with imputed values
    return df

In [14]:
def impute_engineSize(df):
    # Make a copy so the original DataFrame is not modified
    df = df.copy()

     # Calculate median mpg by (model, fuelType)
    model_medians = df.groupby(['model', 'fuelType'])['engineSize'].median()

    # Calculate fallback median mpg by (Brand, fuelType)
    brand_medians = df.groupby(['Brand', 'fuelType'])['engineSize'].median()

    # Calculate fallback median mpg by fuelType
    fuelType_medians = df.groupby(['model'])['engineSize'].median()

    # Iterate through missing rows and fill based on available group
    for idx, row in df[df['engineSize'].isna()].iterrows():
        key_model = (row['model'], row['fuelType'])
        key_brand = (row['Brand'], row['fuelType'])

        if key_model in model_medians.index:
            df.at[idx, 'engineSize'] = model_medians.loc[key_model]
        elif key_brand in brand_medians.index:
            df.at[idx, 'engineSize'] = brand_medians.loc[key_brand]
    df['engineSize'] = df['engineSize'].fillna(df['fuelType'].map(fuelType_medians))
    df['engineSize'] = df['engineSize'].fillna(df['engineSize'].median())
    # Return the DataFrame with imputed values
    return df

In [15]:
def impute_brand (df, min_model_count=20):
    
    # work on a copy
    df = df.copy()

    # Mapping: Modell → häufigste Marke
    df2=df.copy()
    model_to_brand = (
        df2.dropna(subset=['Brand', 'model'])
          .groupby('model')['Brand']
          .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else None)
    )

    # Fehlende Marken füllen, wo Modell bekannt ist
    # sichere Kopie der Masken-Logik, NUR mit df2
    mask = df['Brand'].isna() & df['model'].notna()

    # benutze df2 für das Mapping (niemals df direkt!)
    df.loc[mask, 'Brand'] = df2.loc[mask, 'model'].map(model_to_brand)

    # if still missing, train a classifier
    remaining = df['Brand'].isna().sum()
    if remaining > 0:
        # encode categorical predictors
        predictors = ['transmission', 'engineSize', 'fuelType', 'mpg']
        for col in predictors:
            if df[col].dtype == 'object':
                without_na = df.dropna()
                counts = without_na[col].value_counts()
                df[col] = df[col].map(counts).astype(float)

        # split into known vs missing target
        fuelType_train = df[df['Brand'].notna()]
        test  = df[df['Brand'].isna()]

        X_train = fuelType_train[predictors]
        y_train = fuelType_train['Brand']
        X_test  = test[predictors]

        # encode target
        y_le = LabelEncoder()
        y_train_enc = y_le.fit_transform(y_train.astype(str))

        # train classifier
        clf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=77)
        clf.fit(X_train, y_train_enc)

        # predict and inverse-transform
        preds_enc = clf.predict(X_test)
        preds = y_le.inverse_transform(preds_enc)

        # fill back
        df.loc[df['Brand'].isna(), 'Brand'] = preds

    return df

In [16]:
def impute_model(df, min_brand_count=20):
    # Work on a copy so the original DataFrame is not modified
    df = df.copy()

    
    # --- Train a RandomForest classifier if there are still missing values ---
    # if still missing, train a classifier
    mode_lookup = (
    df.dropna(subset=['Brand', 'transmission', 'model'])
      .groupby(['Brand', 'transmission'])['model']
      .agg(lambda s: s.value_counts().idxmax())
    )
     
    for idx, row in df[df['model'].isna()].iterrows():
        brand = row['Brand']
        trans = row['transmission']
        key = (brand, trans)

        if key in mode_lookup.index:
            # häufigstes Modell einsetzen
            new_value = mode_lookup.loc[key]
            df.at[idx, 'model'] = new_value

    
    remaining = df['model'].isna().sum()
    
    if remaining > 0:
        # encode categorical predictors
        predictors = ['Brand', 'year', 'engineSize', 'mpg', 'tax', 'mileage', 'fuelType', 'transmission']
        for col in predictors:
            if df[col].dtype == 'object':
                without_na = df.dropna()
                counts = without_na[col].value_counts()
                df[col] = df[col].map(counts).astype(float)

        # split into known vs missing target
        fuelType_train = df[df['model'].notna()]
        test  = df[df['model'].isna()]

        X_train = fuelType_train[predictors]
        y_train = fuelType_train['model']
        X_test  = test[predictors]

        # encode target
        y_le = LabelEncoder()
        y_train_enc = y_le.fit_transform(y_train.astype(str))

        # train classifier
        clf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=77)
        clf.fit(X_train, y_train_enc)

        # predict and inverse-transform
        preds_enc = clf.predict(X_test)
        preds = y_le.inverse_transform(preds_enc)

        # fill back
        df.loc[df['model'].isna(), 'model'] = preds

    return df

In [17]:
%%time
X_train["transmission"] = impute_transmission(X_train)["transmission"]
X_train["fuelType"] = impute_fuelType(X_train)["fuelType"]
X_train["engineSize"] = impute_engineSize(X_train)["engineSize"]
X_train["mpg"] = impute_mpg(X_train)["mpg"]
X_train["tax"] = impute_tax(X_train)["tax"]
X_train["year"] = impute_year(X_train)["year"].round().astype(int)
X_train["mileage"] = impute_mileage(X_train)["mileage"]
X_train["paintQuality%"] = impute_paintQuality(X_train)["paintQuality%"]
X_train["previousOwners"] = impute_previousOwners(X_train)["previousOwners"]
X_train["Brand"] = impute_brand(X_train)["Brand"]
X_train["hasDamage"] = X_train["hasDamage"].fillna(1)
X_train["model"] = impute_model(X_train)["model"]

CPU times: user 11.6 s, sys: 313 ms, total: 11.9 s
Wall time: 12.5 s


In [18]:
# quick check
# Look at the Missing values
missing_values = X_train.isnull().sum()

print("Missing values in train:")
print(missing_values[missing_values > 0]) 
print("\n")

Missing values in train:
Series([], dtype: int64)




In [19]:
train_info = X_train.copy()
train_info["is_train"] = True
train_info = train_info.dropna()
train_info["row_id"] = np.nan
X_val["is_train"]=False
X_val["row_id"]=X_val.index
X_val.info()
train_info.info()

<class 'pandas.core.frame.DataFrame'>
Index: 18617 entries, 38667 to 23133
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Brand           18591 non-null  object 
 1   model           18192 non-null  object 
 2   year            18509 non-null  float64
 3   transmission    18065 non-null  object 
 4   mileage         18196 non-null  float64
 5   fuelType        18251 non-null  object 
 6   tax             16588 non-null  float64
 7   mpg             16363 non-null  float64
 8   engineSize      18110 non-null  float64
 9   paintQuality%   18142 non-null  float64
 10  previousOwners  18154 non-null  float64
 11  hasDamage       18232 non-null  float64
 12  is_train        18617 non-null  bool   
 13  row_id          18617 non-null  int64  
dtypes: bool(1), float64(8), int64(1), object(4)
memory usage: 2.0+ MB
<class 'pandas.core.frame.DataFrame'>
Index: 55850 entries, 60965 to 31204
Data columns (total 14 col

In [20]:
%%time
# compute transmission
compute_transmission = pd.concat([train_info.copy(), X_val.loc[X_val["transmission"].isna()]])
transmission_values = impute_transmission(compute_transmission).loc[compute_transmission["is_train"]== False]
for idx, row in X_val[X_val['transmission'].isna()].iterrows():
    row_id = row['row_id']
    new_value = transmission_values.loc[transmission_values['row_id']==row_id, 'transmission'].iloc[0]
    X_val.at[idx, 'transmission'] = new_value

CPU times: user 2.4 s, sys: 96.3 ms, total: 2.5 s
Wall time: 2.71 s


In [21]:
%%time
# compute fuelType
compute_fuelType = pd.concat([train_info.copy(), X_val.loc[X_val["fuelType"].isna()]])
fuelType_values = impute_fuelType(compute_fuelType).loc[compute_fuelType["is_train"]== False]
for idx, row in X_val[X_val['fuelType'].isna()].iterrows():
    row_id = row['row_id']
    new_value = fuelType_values.loc[fuelType_values['row_id']==row_id, 'fuelType'].iloc[0]
    X_val.at[idx, 'fuelType'] = new_value

CPU times: user 2.25 s, sys: 85.4 ms, total: 2.33 s
Wall time: 2.5 s


In [22]:
%%time
# compute engineSize
compute_engineSize = pd.concat([train_info.copy(), X_val.loc[X_val["engineSize"].isna()]])
engineSize_values = impute_engineSize(compute_engineSize).loc[compute_engineSize["is_train"]== False]
for idx, row in X_val[X_val['engineSize'].isna()].iterrows():
    row_id = row['row_id']
    new_value = engineSize_values.loc[engineSize_values['row_id']==row_id, 'engineSize'].iloc[0]
    X_val.at[idx, 'engineSize'] = new_value

CPU times: user 96.4 ms, sys: 9.86 ms, total: 106 ms
Wall time: 107 ms


In [23]:
# fill has damage
X_val["hasDamage"] = X_val["hasDamage"].fillna(1)

In [24]:
%%time
# compute mpg
compute_mpg = pd.concat([train_info.copy(), X_val.loc[X_val["mpg"].isna()]])
mpg_values = impute_mpg(compute_mpg).loc[compute_mpg["is_train"]== False]
for idx, row in X_val[X_val['mpg'].isna()].iterrows():
    row_id = row['row_id']
    new_value = mpg_values.loc[mpg_values['row_id']==row_id, 'mpg'].iloc[0]
    X_val.at[idx, 'mpg'] = new_value

CPU times: user 338 ms, sys: 7.02 ms, total: 345 ms
Wall time: 345 ms


In [25]:
%%time
# compute tax
compute_tax = pd.concat([train_info.copy(), X_val.loc[X_val["tax"].isna()]])
tax_values = impute_tax(compute_tax).loc[compute_tax["is_train"]== False]
for idx, row in X_val[X_val['tax'].isna()].iterrows():
    row_id = row['row_id']
    new_value = tax_values.loc[tax_values['row_id']==row_id, 'tax'].iloc[0]
    X_val.at[idx, 'tax'] = new_value

CPU times: user 403 ms, sys: 9.61 ms, total: 413 ms
Wall time: 414 ms


In [26]:
%%time
# compute year
compute_year = pd.concat([train_info.copy(), X_val.loc[X_val["year"].isna()]])
year_values = impute_year(compute_year).loc[compute_year["is_train"]== False]
for idx, row in X_val[X_val['year'].isna()].iterrows():
    row_id = row['row_id']
    new_value = year_values.loc[year_values['row_id']==row_id, 'year'].iloc[0]
    X_val.at[idx, 'year'] = new_value

CPU times: user 24.9 ms, sys: 4.93 ms, total: 29.8 ms
Wall time: 29.8 ms


In [27]:
%%time
# compute mileage
compute_mileage = pd.concat([train_info.copy(), X_val.loc[X_val["mileage"].isna()]])
mileage_values = impute_mileage(compute_mileage).loc[compute_mileage["is_train"]== False]
for idx, row in X_val[X_val['mileage'].isna()].iterrows():
    row_id = row['row_id']
    new_value = mileage_values.loc[mileage_values['row_id']==row_id, 'mileage'].iloc[0]
    X_val.at[idx, 'mileage'] = new_value

CPU times: user 63.3 ms, sys: 6.84 ms, total: 70.2 ms
Wall time: 72.9 ms


In [28]:
%%time
# compute paintQuality%
compute_paintQuality = pd.concat([train_info.copy(), X_val.loc[X_val["paintQuality%"].isna()]])
paintQuality_values = impute_paintQuality(compute_paintQuality).loc[compute_paintQuality["is_train"]== False]
for idx, row in X_val[X_val['paintQuality%'].isna()].iterrows():
    row_id = row['row_id']
    new_value = paintQuality_values.loc[paintQuality_values['row_id']==row_id, 'paintQuality%'].iloc[0]
    X_val.at[idx, 'paintQuality%'] = new_value

CPU times: user 92.6 ms, sys: 13.6 ms, total: 106 ms
Wall time: 130 ms


In [29]:
%%time
# compute previousOwners
compute_previousOwners = pd.concat([train_info.copy(), X_val.loc[X_val["previousOwners"].isna()]])
previousOwners_values = impute_previousOwners(compute_previousOwners).loc[compute_previousOwners["is_train"]== False]
for idx, row in X_val[X_val['previousOwners'].isna()].iterrows():
    row_id = row['row_id']
    new_value = previousOwners_values.loc[previousOwners_values['row_id']==row_id, 'previousOwners'].iloc[0]
    X_val.at[idx, 'previousOwners'] = new_value

CPU times: user 66.6 ms, sys: 9.01 ms, total: 75.6 ms
Wall time: 79.3 ms


In [30]:
%%time
# compute Brand
compute_brand = pd.concat([train_info.copy(), X_val.loc[X_val["Brand"].isna()]])
brand_values = impute_brand(compute_brand).loc[compute_brand["is_train"]== False]
for idx, row in X_val[X_val['Brand'].isna()].iterrows():
    row_id = row['row_id']
    new_value = brand_values.loc[brand_values['row_id']==row_id, 'Brand'].iloc[0]
    X_val.at[idx, 'Brand'] = new_value

CPU times: user 2.57 s, sys: 99.1 ms, total: 2.67 s
Wall time: 2.95 s


In [31]:
%%time
# compute model
compute_model = pd.concat([train_info.copy(), X_val.loc[X_val["model"].isna()]])
model_values = impute_model(compute_model).loc[compute_model["is_train"]== False]
for idx, row in X_val[X_val['model'].isna()].iterrows():
    row_id = row['row_id']
    new_value = model_values.loc[model_values['row_id']==row_id, 'model'].iloc[0]
    X_val.at[idx, 'model'] = new_value

CPU times: user 88.2 ms, sys: 11.6 ms, total: 99.7 ms
Wall time: 102 ms


In [32]:
X_val = X_val.drop(columns=["row_id", "is_train"])

In [33]:
# quick check
# Look at the Missing values
missing_values = X_train.isnull().sum()

print("Missing values in train:")
print(missing_values[missing_values > 0]) 
print("\n")
missing_values = X_val.isnull().sum()

print("Missing values in validation:")
print(missing_values[missing_values > 0]) 

Missing values in train:
Series([], dtype: int64)


Missing values in validation:
Series([], dtype: int64)


In [34]:
train_info = X_train.copy()
train_info["is_train"] = True
train_info = train_info.dropna()
train_info["row_id"] = np.nan
X_test["is_train"]=False
X_test["row_id"]=X_test.index
X_test.info()
train_info.info()

<class 'pandas.core.frame.DataFrame'>
Index: 32567 entries, 89856 to 99627
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Brand           32525 non-null  object 
 1   model           31833 non-null  object 
 2   year            31560 non-null  float64
 3   transmission    31599 non-null  object 
 4   mileage         31708 non-null  float64
 5   fuelType        31911 non-null  object 
 6   tax             29098 non-null  float64
 7   mpg             28727 non-null  float64
 8   engineSize      31692 non-null  float64
 9   paintQuality%   31774 non-null  float64
 10  previousOwners  31631 non-null  float64
 11  hasDamage       31970 non-null  float64
 12  is_train        32567 non-null  bool   
 13  row_id          32567 non-null  int64  
dtypes: bool(1), float64(8), int64(1), object(4)
memory usage: 3.5+ MB
<class 'pandas.core.frame.DataFrame'>
Index: 55850 entries, 60965 to 31204
Data columns (total 14 col

In [35]:
%%time
# compute transmission
compute_transmission = pd.concat([train_info.copy(), X_test.loc[X_test["transmission"].isna()]])
transmission_values = impute_transmission(compute_transmission).loc[compute_transmission["is_train"]== False]
for idx, row in X_test[X_test['transmission'].isna()].iterrows():
    row_id = row['row_id']
    new_value = transmission_values.loc[transmission_values['row_id']==row_id, 'transmission'].iloc[0]
    X_test.at[idx, 'transmission'] = new_value

CPU times: user 2.39 s, sys: 54.2 ms, total: 2.44 s
Wall time: 2.51 s


In [36]:
%%time
# compute fuelType
compute_fuelType = pd.concat([train_info.copy(), X_test.loc[X_test["fuelType"].isna()]])
fuelType_values = impute_fuelType(compute_fuelType).loc[compute_fuelType["is_train"]== False]
for idx, row in X_test[X_test['fuelType'].isna()].iterrows():
    row_id = row['row_id']
    new_value = fuelType_values.loc[fuelType_values['row_id']==row_id, 'fuelType'].iloc[0]
    X_test.at[idx, 'fuelType'] = new_value


CPU times: user 2.2 s, sys: 54.3 ms, total: 2.25 s
Wall time: 2.26 s


In [37]:
%%time
# compute engineSize
compute_engineSize = pd.concat([train_info.copy(), X_test.loc[X_test["engineSize"].isna()]])
engineSize_values = impute_engineSize(compute_engineSize).loc[compute_engineSize["is_train"]== False]
for idx, row in X_test[X_test['engineSize'].isna()].iterrows():
    row_id = row['row_id']
    new_value = engineSize_values.loc[engineSize_values['row_id']==row_id, 'engineSize'].iloc[0]
    X_test.at[idx, 'engineSize'] = new_value

CPU times: user 148 ms, sys: 5.57 ms, total: 154 ms
Wall time: 153 ms


In [38]:
# fill has damage
X_test["hasDamage"] = X_test["hasDamage"].fillna(1)

In [39]:
%%time
# compute mpg
compute_mpg = pd.concat([train_info.copy(), X_test.loc[X_test["mpg"].isna()]])
mpg_values = impute_mpg(compute_mpg).loc[compute_mpg["is_train"]== False]
for idx, row in X_test[X_test['mpg'].isna()].iterrows():
    row_id = row['row_id']
    new_value = mpg_values.loc[mpg_values['row_id']==row_id, 'mpg'].iloc[0]
    X_test.at[idx, 'mpg'] = new_value

CPU times: user 689 ms, sys: 22.7 ms, total: 711 ms
Wall time: 801 ms


In [40]:
%%time
# compute tax
compute_tax = pd.concat([train_info.copy(), X_test.loc[X_test["tax"].isna()]])
tax_values = impute_tax(compute_tax).loc[compute_tax["is_train"]== False]
for idx, row in X_test[X_test['tax'].isna()].iterrows():
    row_id = row['row_id']
    new_value = tax_values.loc[tax_values['row_id']==row_id, 'tax'].iloc[0]
    X_test.at[idx, 'tax'] = new_value

CPU times: user 714 ms, sys: 52.6 ms, total: 767 ms
Wall time: 808 ms


In [41]:
%%time
# compute year
compute_year = pd.concat([train_info.copy(), X_test.loc[X_test["year"].isna()]])
year_values = impute_year(compute_year).loc[compute_year["is_train"]== False]
for idx, row in X_test[X_test['year'].isna()].iterrows():
    row_id = row['row_id']
    new_value = year_values.loc[year_values['row_id']==row_id, 'year'].iloc[0]
    X_test.at[idx, 'year'] = new_value

CPU times: user 151 ms, sys: 9.41 ms, total: 161 ms
Wall time: 216 ms


In [42]:
%%time
# compute mileage
compute_mileage = pd.concat([train_info.copy(), X_test.loc[X_test["mileage"].isna()]])
mileage_values = impute_mileage(compute_mileage).loc[compute_mileage["is_train"]== False]
for idx, row in X_test[X_test['mileage'].isna()].iterrows():
    row_id = row['row_id']
    new_value = mileage_values.loc[mileage_values['row_id']==row_id, 'mileage'].iloc[0]
    X_test.at[idx, 'mileage'] = new_value

CPU times: user 105 ms, sys: 5.43 ms, total: 111 ms
Wall time: 110 ms


In [43]:
%%time
# compute paintQuality%
compute_paintQuality = pd.concat([train_info.copy(), X_test.loc[X_test["paintQuality%"].isna()]])
paintQuality_values = impute_paintQuality(compute_paintQuality).loc[compute_paintQuality["is_train"]== False]
for idx, row in X_test[X_test['paintQuality%'].isna()].iterrows():
    row_id = row['row_id']
    new_value = paintQuality_values.loc[paintQuality_values['row_id']==row_id, 'paintQuality%'].iloc[0]
    X_test.at[idx, 'paintQuality%'] = new_value

CPU times: user 97.1 ms, sys: 5.88 ms, total: 103 ms
Wall time: 103 ms


In [44]:
%%time
# compute previousOwners
compute_previousOwners = pd.concat([train_info.copy(), X_test.loc[X_test["previousOwners"].isna()]])
previousOwners_values = impute_previousOwners(compute_previousOwners).loc[compute_previousOwners["is_train"]== False]
for idx, row in X_test[X_test['previousOwners'].isna()].iterrows():
    row_id = row['row_id']
    new_value = previousOwners_values.loc[previousOwners_values['row_id']==row_id, 'previousOwners'].iloc[0]
    X_test.at[idx, 'previousOwners'] = new_value

CPU times: user 110 ms, sys: 5.48 ms, total: 115 ms
Wall time: 113 ms


In [45]:
%%time
# compute Brand
compute_brand = pd.concat([train_info.copy(), X_test.loc[X_test["Brand"].isna()]])
brand_values = impute_brand(compute_brand).loc[compute_brand["is_train"]== False]
for idx, row in X_test[X_test['Brand'].isna()].iterrows():
    row_id = row['row_id']
    new_value = brand_values.loc[brand_values['row_id']==row_id, 'Brand'].iloc[0]
    X_test.at[idx, 'Brand'] = new_value

CPU times: user 2.45 s, sys: 73.9 ms, total: 2.52 s
Wall time: 2.6 s


In [46]:
%%time
# compute model
compute_model = pd.concat([train_info.copy(), X_test.loc[X_test["model"].isna()]])
model_values = impute_model(compute_model).loc[compute_model["is_train"]== False]
for idx, row in X_test[X_test['model'].isna()].iterrows():
    row_id = row['row_id']
    new_value = model_values.loc[model_values['row_id']==row_id, 'model'].iloc[0]
    X_test.at[idx, 'model'] = new_value

CPU times: user 138 ms, sys: 8.22 ms, total: 146 ms
Wall time: 147 ms


In [47]:
X_test = X_test.drop(columns=["row_id", "is_train"])

In [48]:
# quick check
# Look at the Missing values
missing_values = X_test.isnull().sum()

print("Missing values in test:")
print(missing_values[missing_values > 0])

Missing values in test:
Series([], dtype: int64)


# Encoding categorical variables

- We have to find out which encoding method is the best for our categorical features

In [49]:
trans_medians = X_train.groupby('transmission')['mpg'].median()
X_train['mpg_median_trans'] = X_train['transmission'].map(trans_medians)
X_val['mpg_median_trans'] = X_val['transmission'].map(trans_medians)
X_test['mpg_median_trans'] = X_test['transmission'].map(trans_medians)

In [50]:
low_card_cat = [c for c in categorical_cols if X_train[c].nunique() <= 15]
high_card_cat = [c for c in categorical_cols if c not in low_card_cat]

ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
ohe.fit(X_train[low_card_cat])

def apply_ohe(df):
    ohe_arr = ohe.transform(df[low_card_cat])
    ohe_cols = ohe.get_feature_names_out(low_card_cat)
    ohe_df = pd.DataFrame(ohe_arr, columns=ohe_cols, index=df.index)
    df_rest = df.drop(columns=low_card_cat)
    return pd.concat([df_rest, ohe_df], axis=1)

x_train_enc = apply_ohe(X_train)
x_val_enc   = apply_ohe(X_val)
x_test_enc  = apply_ohe(X_test)


# print the columns after encoding
print("Columns after encoding:")
print(x_test_enc.columns.tolist())


# Print the number of columns after encoding for test val and train
print("Number of columns after encoding:")
print("x_train_enc:", x_train_enc.shape[1])
print("x_val_enc:", x_val_enc.shape[1])
print("x_test_enc:", x_test_enc.shape[1])

Columns after encoding:
['model', 'year', 'mileage', 'tax', 'mpg', 'engineSize', 'paintQuality%', 'previousOwners', 'hasDamage', 'mpg_median_trans', 'Brand_Audi', 'Brand_BMW', 'Brand_Ford', 'Brand_Hyundai', 'Brand_Mercedes-Benz', 'Brand_Opel', 'Brand_Toyota', 'Brand_Volkswagen', 'Brand_Škoda', 'transmission_Automatic', 'transmission_Manual', 'transmission_Other', 'transmission_Semi-Auto', 'fuelType_Diesel', 'fuelType_Electric', 'fuelType_Hybrid', 'fuelType_Other', 'fuelType_Petrol']
Number of columns after encoding:
x_train_enc: 28
x_val_enc: 28
x_test_enc: 28


# Feature Engineering

In [51]:
x_train_enc['mpg_diff_transmission'] = x_train_enc['mpg'] - x_train_enc['mpg_median_trans']
x_train_enc['mpg_diff_transmission'] = x_train_enc['mpg_diff_transmission'].fillna(0)

x_train_enc = x_train_enc.drop(columns=["mpg_median_trans"])

# age affects cars much more than the year: car_age in years
x_train_enc['car_age'] = 2020 - x_train_enc['year']
# it is important how efficent the motor is, 
x_train_enc['efficiency_ratio'] = x_train_enc['mpg'] / (x_train_enc['engineSize'] + 0.1)

x_train_enc['mileage_per_year'] = x_train_enc['mileage'] / (x_train_enc['car_age'] + 1)

#brand_popularity = x_train_enc['Brand'].value_counts(normalize=True)
#x_train_enc['brand_popularity'] = x_train_enc['Brand'].map(brand_popularity)

x_train_enc['owners_flag'] = (x_train_enc['previousOwners'] > 2).astype(int)

x_train_enc['previousOwners_sq'] = x_train_enc['previousOwners'] ** 2

x_train_enc['engine_tax_ratio'] = x_train_enc['engineSize'] / (x_train_enc['tax'] + 1)

In [52]:
x_train_enc.describe()

,year,mileage,tax,mpg,engineSize,paintQuality%,previousOwners,hasDamage,Brand_Audi,Brand_BMW,Brand_Ford,Brand_Hyundai,Brand_Mercedes-Benz,Brand_Opel,Brand_Toyota,Brand_Volkswagen,Brand_Škoda,transmission_Automatic,transmission_Manual,transmission_Other,transmission_Semi-Auto,fuelType_Diesel,fuelType_Electric,fuelType_Hybrid,fuelType_Other,fuelType_Petrol,mpg_diff_transmission,car_age,efficiency_ratio,mileage_per_year,owners_flag,previousOwners_sq,engine_tax_ratio
count,55850.000000,55850.000000,55850.000000,55850.000000,55850.000000,55850.000000,55850.000000,55850.000000,55850.000000,55850.000000,55850.000000,55850.000000,55850.000000,55850.000000,55850.000000,55850.000000,55850.000000,55850.000000,55850.000000,55850.000000,55850.000000,55850.000000,55850.000000,55850.000000,55850.000000,55850.000000,55850.000000,55850.000000,55850.000000,55850.000000,55850.000000,55850.000000,55850.000000
mean,2017.064082,23353.876714,118.140789,54.841588,1.668882,64.249855,2.011961,0.020340,0.098227,0.098890,0.215721,0.044261,0.158281,0.125855,0.062346,0.138926,0.057493,0.201701,0.568397,0.000072,0.229830,0.412408,0.000036,0.030331,0.002149,0.555076,-0.091383,2.935918,34.255849,5361.039294,0.391406,6.078890,0.130732
std,2.141554,21550.014693,65.062662,11.372138,0.554516,20.379646,1.425111,0.141162,0.297625,0.298517,0.411325,0.205677,0.365008,0.331689,0.241784,0.345872,0.232785,0.401274,0.495304,0.008463,0.420727,0.492272,0.005984,0.171499,0.046304,0.496962,11.147364,2.141554,12.824670,3928.029498,0.488069,6.192531,0.395289
min,2000.000000,1.000000,0.000000,9.429179,1.000000,1.638913,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-48.270821,0.000000,2.284061,0.076923,0.000000,0.000000,0.003021
25%,2016.000000,7500.000000,30.000000,47.100000,1.200000,47.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-7.500000,1.000000,25.681818,2832.500000,0.000000,1.000000,0.009589
50%,2017.000000,17496.500000,145.000000,55.400000,1.600000,64.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,3.000000,31.500000,4861.714286,0.000000,4.000000,0.013245
75%,2019.000000,32475.000000,145.000000,62.800000,2.000000,82.000000,3.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,1.000000,8.000000,4.000000,43.705882,7050.482143,1.000000,9.000000,0.032258
max,2020.000000,323000.000000,580.000000,94.100000,6.200000,99.000000,6.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,41.800000,20.000000,67.545455,101153.348285,1.000000,36.000000,3.822758


In [53]:
x_val_enc['mpg_diff_transmission'] = x_val_enc['mpg'] - x_val_enc['mpg_median_trans']
x_val_enc['mpg_diff_transmission'] = x_val_enc['mpg_diff_transmission'].fillna(0)

x_val_enc = x_val_enc.drop(columns=["mpg_median_trans"])

# age affects cars much more than the year: car_age in years
x_val_enc['car_age'] = 2020 - x_val_enc['year']
# it is important how efficent the motor is, 
x_val_enc['efficiency_ratio'] = x_val_enc['mpg'] / (x_val_enc['engineSize'] + 0.1)

x_val_enc['mileage_per_year'] = x_val_enc['mileage'] / (x_val_enc['car_age'] + 1)

#brand_popularity = x_val_enc['Brand'].value_counts(normalize=True)
#x_val_enc['brand_popularity'] = x_val_enc['Brand'].map(brand_popularity)

x_val_enc['owners_flag'] = (x_val_enc['previousOwners'] > 2).astype(int)

x_val_enc['previousOwners_sq'] = x_val_enc['previousOwners'] ** 2

x_val_enc['engine_tax_ratio'] = x_val_enc['engineSize'] / (x_val_enc['tax'] + 1)

In [54]:
x_test_enc['mpg_diff_transmission'] = x_test_enc['mpg'] - x_test_enc['mpg_median_trans']
x_test_enc['mpg_diff_transmission'] = x_test_enc['mpg_diff_transmission'].fillna(0)

x_test_enc = x_test_enc.drop(columns=["mpg_median_trans"])

# age affects cars much more than the year: car_age in years
x_test_enc['car_age'] = 2020 - x_test_enc['year']
# it is important how efficent the motor is, 
x_test_enc['efficiency_ratio'] = x_test_enc['mpg'] / (x_test_enc['engineSize'] + 0.1)

x_test_enc['mileage_per_year'] = x_test_enc['mileage'] / (x_test_enc['car_age'] + 1)

#brand_popularity = x_test_enc['Brand'].value_counts(normalize=True)
#x_test_enc['brand_popularity'] = x_test_enc['Brand'].map(brand_popularity)

x_test_enc['owners_flag'] = (x_test_enc['previousOwners'] > 2).astype(int)

x_test_enc['previousOwners_sq'] = x_test_enc['previousOwners'] ** 2

x_test_enc['engine_tax_ratio'] = x_test_enc['engineSize'] / (x_test_enc['tax'] + 1)

# Scaling

After encoding all our values are numerical, but we need to scale the features for better model performance.
The Values for mileage can be very high compared to other features, so scaling is important.



In [55]:
scaler = RobustScaler()

# numeric cols after FE (on train)
num_after_enc = x_train_enc.select_dtypes(include=["number"]).columns

# fit on TRAIN only
scaler.fit(x_train_enc[num_after_enc])

# copy
x_train_final = x_train_enc.copy()
x_val_final   = x_val_enc.copy()
x_test_final  = x_test_enc.copy()

# scale train and val on same cols
x_train_final[num_after_enc] = scaler.transform(x_train_enc[num_after_enc])
x_val_final[num_after_enc]   = scaler.transform(x_val_enc[num_after_enc])

# for test: only the intersection of cols
test_cols = [c for c in num_after_enc if c in x_test_enc.columns]
x_test_final[test_cols] = scaler.transform(x_test_enc[test_cols])

In [56]:
# Check for NaN values in each dataset
def check_nan(df, name):
    nan_cols = df.columns[df.isna().any()].tolist()
    if nan_cols:
        print(f"{name} has NaN values in columns: {nan_cols}")
    else:
        print(f"{name} has no NaN values.")


check_nan(x_train_final, "x_train_final")
check_nan(y_train.to_frame(), "y_train")
check_nan(x_val_final, "x_val_final")
check_nan(y_val.to_frame(), "y_val")
check_nan(x_test_final, "x_test_final")

x_train_final has no NaN values.
y_train has no NaN values.
x_val_final has no NaN values.
y_val has no NaN values.
x_test_final has no NaN values.


# Output Save

In [57]:
# Put CarID index back as a column
x_train_final = x_train_final.reset_index().rename(columns={'index': 'CarID'})
x_val_final = x_val_final.reset_index().rename(columns={'index': 'CarID'})
x_test_final = x_test_final.reset_index().rename(columns={'index': 'CarID'})

In [58]:
# Save Processed Datasets
"""
Save X_train, y_train, X_val, y_val, and X_test separately.
This structure is cleaner for later model loading and avoids re-splitting.
"""

import os

output_dir = os.path.join(data_dir, "encoded_data")
os.makedirs(output_dir, exist_ok=True)

# Save feature and target sets separately
x_train_final.to_csv(os.path.join(output_dir, "12_X_train.csv"), index=False)
y_train.to_csv(os.path.join(output_dir, "12_y_train.csv"), index=False)

x_val_final.to_csv(os.path.join(output_dir, "12_X_val.csv"), index=False)
y_val.to_csv(os.path.join(output_dir, "12_y_val.csv"), index=False)

x_test_final.to_csv(os.path.join(output_dir, "12_X_test.csv"), index=False)

print("Processed data saved successfully (X/y separated):")
print(f"X_train: {x_train_final.shape}, y_train: {y_train.shape}")
print(f"X_val:   {x_val_final.shape}, y_val: {y_val.shape}")
print(f"X_test:  {x_test_final.shape}")


Processed data saved successfully (X/y separated):
X_train: (55850, 35), y_train: (55850,)
X_val:   (18617, 35), y_val: (18617,)
X_test:  (32567, 35)
